## Precision Comparison — Ximea vs Prime95B

For molecules that show up in *both* datasets (matched via the transform
fitted in `Registration.ipynb`), compares localisation precision directly
between the two cameras.

**Precision metric**: for each matched molecule, on each camera, the
empirical scatter of its own raw per-frame localisations around its own mean
position — `sigma_xy = sqrt((std(dx)^2 + std(dy)^2) / 2) * pixel_size_nm`,
using the `*_linked_frames.h5` tables (one row per raw localisation, grouped
by `molecular_index`) from the two PostAnalysis notebooks. This is the same
approach validated earlier for `z_defocus_analysis.ipynb`, preferred over
trusting the fitted `xc_err`/`yc_err` uncertainty alone.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
from pathlib import Path

import sys
sys.path.append('../..')

from src import IOFunctions
from src.RegistrationFunctions import match_spot_pairs_indexed, load_transform

IO = IOFunctions.IO_Functions()


In [ ]:
# ── Paths and parameters ────────────────────────────────────────────────────────
XIMEA_FOLDER    = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads/100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Ximea')
PRIME95B_FOLDER = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads/100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Prime95B')
TRANSFORM_PATH  = XIMEA_FOLDER.parent / 'ximea_to_prime95b_transform.csv'

PIXEL_SIZE_XIMEA_NM    = 69.0
PIXEL_SIZE_PRIME95B_NM = 110.0

# Tight match distance for the FINAL pairing — points are already registered
# at this point, so this should be close to the expected residual registration
# error (see Registration.ipynb's residual histogram), not the generous
# first-pass distance used during transform fitting.
FINAL_MATCH_DISTANCE_NM = 150.0

POS_RE = re.compile(r'Pos-\d+-\d+')


def fov_positions(folder, suffix):
    out = {}
    for p in sorted(folder.glob(f'*{suffix}')):
        m = POS_RE.search(p.name)
        if m:
            out[m.group()] = p
    return out


tform = load_transform(str(TRANSFORM_PATH))
print(f'Loaded transform: scale={tform.scale}  rotation={np.degrees(tform.rotation):.3f} deg  '
      f'translation={tform.translation}')

ximea_sm_pos     = fov_positions(XIMEA_FOLDER, '_linked_sm.h5')
ximea_frames_pos = fov_positions(XIMEA_FOLDER, '_linked_frames.h5')
prime95b_sm_pos     = fov_positions(PRIME95B_FOLDER, '_linked_sm.h5')
prime95b_frames_pos = fov_positions(PRIME95B_FOLDER, '_linked_frames.h5')
common_pos = sorted(set(ximea_sm_pos) & set(prime95b_sm_pos))
print(f'{len(common_pos)} shared Pos-i-j FOVs')


In [ ]:
# ── Empirical sigma_xy per linked molecule ──────────────────────────────────────
def sigma_xy_per_molecule(single_frame_db, pixel_size_nm):
    """Empirical localisation precision per molecular_index: RMS scatter of its
    raw per-frame (xc, yc) around its own mean, in nm."""
    rows = []
    for mol_id, g in single_frame_db.groupby('molecular_index'):
        if len(g) < 2:
            continue
        dx = g['xc'] - g['xc'].mean()
        dy = g['yc'] - g['yc'].mean()
        sigma_xy = np.sqrt((np.var(dx, ddof=1) + np.var(dy, ddof=1)) / 2) * pixel_size_nm
        rows.append({
            'molecular_index': mol_id,
            'xc': g['xc'].mean(), 'yc': g['yc'].mean(),
            'sigma_xy_nm': sigma_xy,
            'n_locs': len(g),
            'mean_photons': g['photons'].mean() if 'photons' in g.columns else np.nan,
        })
    return pd.DataFrame(rows)


print('sigma_xy_per_molecule helper defined.')


In [ ]:
# ── Match molecules per shared FOV and compare precision ───────────────────────
paired_rows = []

for pos in common_pos:
    ximea_sm     = IO.read_h5_database(str(ximea_sm_pos[pos]))
    ximea_frames = IO.read_h5_database(str(ximea_frames_pos[pos]))
    prime95b_sm     = IO.read_h5_database(str(prime95b_sm_pos[pos]))
    prime95b_frames = IO.read_h5_database(str(prime95b_frames_pos[pos]))

    ximea_prec    = sigma_xy_per_molecule(ximea_frames, PIXEL_SIZE_XIMEA_NM)
    prime95b_prec = sigma_xy_per_molecule(prime95b_frames, PIXEL_SIZE_PRIME95B_NM)
    if len(ximea_prec) == 0 or len(prime95b_prec) == 0:
        continue

    # Register Ximea (xc, yc) [px] -> nm -> Prime95B-frame nm, then match against
    # Prime95B (xc, yc) [px] -> nm, both in the SAME shared-FOV local coordinate frame.
    ximea_nm      = ximea_prec[['xc', 'yc']].to_numpy() * PIXEL_SIZE_XIMEA_NM
    ximea_reg_nm  = tform(ximea_nm)
    prime95b_nm   = prime95b_prec[['xc', 'yc']].to_numpy() * PIXEL_SIZE_PRIME95B_NM

    src_idx, dst_idx = [], []
    if len(ximea_reg_nm) and len(prime95b_nm):
        src_idx, dst_idx = match_spot_pairs_indexed(ximea_reg_nm, prime95b_nm, max_distance=FINAL_MATCH_DISTANCE_NM)

    for i, j in zip(src_idx, dst_idx):
        paired_rows.append({
            'pos': pos,
            'sigma_xy_ximea_nm':    ximea_prec.iloc[i]['sigma_xy_nm'],
            'sigma_xy_prime95b_nm': prime95b_prec.iloc[j]['sigma_xy_nm'],
            'photons_ximea':        ximea_prec.iloc[i]['mean_photons'],
            'photons_prime95b':     prime95b_prec.iloc[j]['mean_photons'],
            'n_locs_ximea':         ximea_prec.iloc[i]['n_locs'],
            'n_locs_prime95b':      prime95b_prec.iloc[j]['n_locs'],
        })

    print(f'{pos}: Ximea={len(ximea_prec):3d} molecules, Prime95B={len(prime95b_prec):3d} molecules, '
          f'matched={len(src_idx):3d}')

paired_df = pd.DataFrame(paired_rows)
print(f'\nTotal matched molecule pairs across all FOVs: {len(paired_df)}')


In [ ]:
# ── Direct precision comparison for matched molecules ───────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))

lim = float(np.nanpercentile(
    np.concatenate([paired_df['sigma_xy_ximea_nm'], paired_df['sigma_xy_prime95b_nm']]), 99
)) * 1.1

axs[0].scatter(paired_df['sigma_xy_ximea_nm'], paired_df['sigma_xy_prime95b_nm'], s=12, alpha=0.5)
axs[0].plot([0, lim], [0, lim], 'k--', lw=1, label='y = x')
axs[0].set_xlim(0, lim); axs[0].set_ylim(0, lim)
axs[0].set_xlabel(r'$\sigma_{xy}$ Ximea / nm')
axs[0].set_ylabel(r'$\sigma_{xy}$ Prime95B / nm')
axs[0].set_title(f'Matched-molecule precision (n={len(paired_df)})')
axs[0].legend()
axs[0].set_aspect('equal')

bins = np.linspace(0, lim, 40)
axs[1].hist(paired_df['sigma_xy_ximea_nm'], bins=bins, alpha=0.5, label='Ximea')
axs[1].hist(paired_df['sigma_xy_prime95b_nm'], bins=bins, alpha=0.5, label='Prime95B')
axs[1].set_xlabel(r'$\sigma_{xy}$ / nm'); axs[1].set_ylabel('Count'); axs[1].legend()
axs[1].set_title('Precision distributions (matched molecules only)')

plt.tight_layout()
plt.show()

print(f"Median sigma_xy — Ximea: {paired_df['sigma_xy_ximea_nm'].median():.1f} nm, "
      f"Prime95B: {paired_df['sigma_xy_prime95b_nm'].median():.1f} nm")


In [ ]:
# ── Precision vs photon count, per camera (context for the paired comparison) ──
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(paired_df['photons_ximea'], paired_df['sigma_xy_ximea_nm'], s=12, alpha=0.5, label='Ximea')
ax.scatter(paired_df['photons_prime95b'], paired_df['sigma_xy_prime95b_nm'], s=12, alpha=0.5, label='Prime95B')
ax.set_xscale('log')
ax.set_xlabel('Mean photons per localisation')
ax.set_ylabel(r'$\sigma_{xy}$ / nm')
ax.legend()
ax.set_title('Precision vs photon count (matched molecules)')
plt.tight_layout()
plt.show()
